### Mass model the PCA sets and output to files

In [15]:
import pandas as pd
import xgboost as xgb
import ast
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import matplotlib as plt
import seaborn as sns
import os

In [16]:
# Parse dataset parameters
parameters = pd.read_csv(rf'../dataset/best_parameter_for_pca_permutations_2026-09-21_15-25-40.csv')
parameters['parameters_dict'] = parameters['parameters'].apply(ast.literal_eval)

# Ensure results folder exists
results_dir = '../dataset/results'
os.makedirs(results_dir, exist_ok=True)

for i, row in enumerate(parameters.itertuples(index=True)):
    file_name = row.dataset_name
    best_parameters = row.parameters_dict

    dataset_path = os.path.join('../dataset', file_name)
    dataset = pd.read_csv(dataset_path)

    model = xgb.XGBClassifier(
        **best_parameters,
        objective='multi:softprob',
        eval_metric='mlogloss',
        tree_method='hist',
        random_state=69,
        n_jobs=-1
    )

    # Slice PCA feature columns safely
    column_count = len(dataset.columns.tolist()) - 2
    X = dataset[[f"PC{k}" for k in range(1, column_count + 1)]]

    le = LabelEncoder()
    y = le.fit_transform(dataset['Activity'])
    groups = dataset['subject']

    # Unseen subject split
    gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=69)
    train_idx, test_idx = next(gss.split(X, y, groups=groups))

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    groups_test = groups.iloc[test_idx]

    # Train model
    model.fit(X_train, y_train)

    # Predict & decode labels
    y_pred = model.predict(X_test)

    y_test_decoded = le.inverse_transform(y_test)
    y_pred_decoded = le.inverse_transform(y_pred)

    # Construct DataFrame using clean 1D arrays
    predictions_df = pd.DataFrame({
        'Subject': groups_test.values,
        'Actual_Activity': y_test_decoded,
        'Predicted_Activity': y_pred_decoded
    })

    # Save cleanly without index column
    predictions_df_name = f"pred_{file_name.replace('.csv', '')}_xgboost.csv"
    predictions_df.to_csv(os.path.join(results_dir, predictions_df_name), index=False)

    print(f'No. {i + 1} / {6} is done!')

No. 1 / 6 is done!
No. 2 / 6 is done!
No. 3 / 6 is done!
No. 4 / 6 is done!
No. 5 / 6 is done!
No. 6 / 6 is done!
